# Interactive Suitability Map — Jonglei State, South Sudan
### Run Cell 1 to load data, then Cell 2 to launch the app.

In [7]:
import os, gc, io
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds as bounds_window
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, Image as IPImage

# ── Working directory ─────────────────────────────────────────
PROJECT_DIR = r'C:\Users\Owner\anaconda_projects\f7a0b3b6-6018-428d-8b28-da481efada7b'
os.chdir(PROJECT_DIR)

# ── Reference grid from DEM ───────────────────────────────────
SCALE = 0.10
with rasterio.open('Reclass_DEM1_2.tif') as _ref:
    REF_H      = max(1, int(_ref.height * SCALE))
    REF_W      = max(1, int(_ref.width  * SCALE))
    REF_BOUNDS = _ref.bounds

# ── Load layer: clip to DEM bounds, resample to display grid ──
def load_layer(path):
    with rasterio.open(path) as src:
        win = bounds_window(
            REF_BOUNDS.left, REF_BOUNDS.bottom,
            REF_BOUNDS.right, REF_BOUNDS.top,
            src.transform
        )
        d = src.read(1, window=win, out_shape=(REF_H, REF_W),
                     resampling=Resampling.nearest).astype(np.float32)
        nd = src.nodata
    if nd is not None:
        d[d == nd] = np.nan
    d = np.round(d)
    d[(d < 1) | (d > 5)] = np.nan
    return d

print('Loading all 11 layers...')
fhi_layers = {
    'Elevation':   load_layer('Reclass_DEM1_2.tif'),
    'Slope':       load_layer('Reclass_Slop1.tif'),
    'TWI':         load_layer('Reclass_TWI1.tif'),
    'DistRiver':   load_layer('Reclass_EucD1.tif'),
    'Rainfall':    load_layer('Reclass_Rain1.tif'),
    'LULC':        load_layer('Reclass_LULC_f1.tif'),
    'SoilTexture': load_layer('Reclass_soil1.tif'),
}
svi_layers = {
    'DistRoad':      load_layer('Reclass_road1.tif'),
    'DistHealth':    load_layer('Reclass_Dist_to_health1.tif'),
    'PopDensity':    load_layer('Reclass_population1.tif'),
    'ConflictIndex': load_layer('Reclass_Conflict1.tif'),
}
gc.collect()
print(f'Done.  Display grid: {REF_H} x {REF_W}  |  Underlying cell size: ~30 m x 30 m')

Loading all 11 layers...
Done.  Display grid: 1367 x 1734  |  Underlying cell size: ~30 m x 30 m


In [8]:
from IPython.display import HTML as IPHTML
import pathlib

# ── Weight constraints ────────────────────────────────────────
MIN_W = 1.0    # no single factor can drop below 1 %
MAX_W = 60.0   # no single factor can exceed 60 %

# ── Default weights ───────────────────────────────────────────
FHI_DEFAULTS = {
    'Elevation':   18.09,
    'Slope':       18.29,
    'TWI':         17.48,
    'DistRiver':   14.23,
    'Rainfall':    12.73,
    'LULC':        11.95,
    'SoilTexture':  7.24,
}
SVI_DEFAULTS = {
    'DistRoad':      25.0,
    'DistHealth':    25.0,
    'PopDensity':    25.0,
    'ConflictIndex': 25.0,
}

# ── Weight panel builder ──────────────────────────────────────
def build_weight_panel(defaults):
    state = {'busy': False}
    sliders, texts = {}, {}

    for name, val in defaults.items():
        sliders[name] = widgets.FloatSlider(
            value=val, min=MIN_W, max=MAX_W, step=0.01,
            description=f'{name}:',
            continuous_update=False,
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px'),
            readout=False,
        )
        texts[name] = widgets.BoundedFloatText(
            value=val, min=MIN_W, max=MAX_W, step=0.01,
            layout=widgets.Layout(width='72px'),
        )

    total_lbl = widgets.HTML(
        value=f'<span style="font-size:13px;font-weight:600;color:#333">'
              f'Total: {sum(defaults.values()):.2f} %</span>'
    )
    reset_btn = widgets.Button(
        description='↺  Reset',
        tooltip='Restore methodology defaults',
        button_style='warning',
        layout=widgets.Layout(width='90px', height='28px'),
    )

    def do_adjust(changed_name, new_val, old_val):
        if state['busy']:
            return
        state['busy'] = True
        try:
            delta = new_val - old_val
            others = {k: sliders[k] for k in sliders if k != changed_name}
            tot = sum(s.value for s in others.values())
            if tot > 1e-6 and abs(delta) > 1e-6:
                for k, s in others.items():
                    raw = s.value - delta * s.value / tot
                    s.value = max(MIN_W, min(MAX_W, round(raw, 2)))
                    texts[k].value = s.value
            total_lbl.value = (
                f'<span style="font-size:13px;font-weight:600;color:#333">'
                f'Total: {sum(s.value for s in sliders.values()):.2f} %</span>'
            )
        finally:
            state['busy'] = False

    def make_slider_cb(n):
        def cb(change):
            if state['busy']:
                return
            texts[n].value = change['new']
            do_adjust(n, change['new'], change['old'])
        return cb

    def make_text_cb(n):
        def cb(change):
            if state['busy']:
                return
            sliders[n].value = change['new']
        return cb

    for n in sliders:
        sliders[n].observe(make_slider_cb(n), names='value')
        texts[n].observe(make_text_cb(n),     names='value')

    def on_reset(b):
        state['busy'] = True
        try:
            for n, v in defaults.items():
                sliders[n].value = v
                texts[n].value   = v
            total_lbl.value = (
                f'<span style="font-size:13px;font-weight:600;color:#333">'
                f'Total: {sum(defaults.values()):.2f} %</span>'
            )
        finally:
            state['busy'] = False

    reset_btn.on_click(on_reset)

    rows = []
    for name in defaults:
        pct = widgets.HTML(
            '<span style="font-size:12px;color:#666;margin-left:2px">%</span>'
        )
        rows.append(widgets.HBox(
            [sliders[name], texts[name], pct],
            layout=widgets.Layout(align_items='center', margin='2px 0'),
        ))

    note = widgets.HTML(
        f'<span style="font-size:11px;color:#999">'
        f'Each factor: {MIN_W}% – {MAX_W}%  ·  adjusts others proportionally</span>'
    )
    footer = widgets.HBox(
        [total_lbl, reset_btn],
        layout=widgets.Layout(justify_content='space-between',
                              margin='8px 4px 0', align_items='center'),
    )
    panel = widgets.VBox(
        rows + [note, footer],
        layout=widgets.Layout(padding='12px 16px', border='1px solid #d0d0d0',
                              margin='2px'),
    )
    return sliders, panel


# ── Suitability classification ────────────────────────────────
def categorize_si(si):
    cats = np.full(si.shape, '', dtype=object)
    v = ~np.isnan(si)
    cats[v & (si <= 1.5)]               = 'Very Low Suitability'
    cats[v & (si > 1.5) & (si <= 2.5)] = 'Low Suitability'
    cats[v & (si > 2.5) & (si <= 3.5)] = 'Moderate Suitability'
    cats[v & (si > 3.5) & (si <= 4.5)] = 'High Suitability'
    cats[v & (si > 4.5)]               = 'Very High Suitability'
    return cats


# ── PNG helper — auto-scales colours to data range ───────────
# vmin/vmax=None → matplotlib auto-scales (matches FHI_SVI.ipynb appearance)
# Pass vmin=1,vmax=5 only for SI where fixed scale is needed
def mpl_png(arr, title, cmap, cb_label, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(5, 4), dpi=90)
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.65, label=cb_label)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=90)
    plt.close(fig)
    buf.seek(0)
    return widgets.Image(value=buf.read(), format='png')


# ── Build plotly SI figure (reused for display and export) ────
def build_si_figure(si, cats):
    fig = go.Figure(go.Heatmap(
        z=si.tolist(),
        colorscale='RdYlGn',
        zmin=1, zmax=5,
        zsmooth=False,
        text=cats.tolist(),
        hovertemplate=(
            '<b>Score: %{z:.2f} / 5</b><br>'
            '%{text}<br>'
            '<span style="font-size:11px;color:#888">~30 m × 30 m cell</span>'
            '<extra></extra>'
        ),
        colorbar=dict(
            title=dict(text='Suitability', side='right', font=dict(size=13)),
            tickvals=[1, 2, 3, 4, 5],
            ticktext=['1  Very Low', '2  Low', '3  Moderate',
                      '4  High', '5  Very High'],
            tickfont=dict(size=11),
            thickness=18, len=0.85,
        ),
    ))
    fig.update_layout(
        title=dict(
            text=('Suitability Index — Jonglei State, South Sudan<br>'
                  '<sup style="color:#666">FHI: 60 %  |  SVI: 40 %  |  '
                  '~30 m × 30 m cells  |  hover for score</sup>'),
            x=0.5, font=dict(size=15, color='#222'),
        ),
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   autorange='reversed'),
        margin=dict(l=10, r=10, t=75, b=10),
        height=600,
        hoverlabel=dict(bgcolor='white', bordercolor='#aaa',
                        font=dict(size=13, color='#222')),
        paper_bgcolor='white', plot_bgcolor='white',
    )
    return fig


# ── Build tabbed weight panels ────────────────────────────────
fhi_sliders, fhi_panel = build_weight_panel(FHI_DEFAULTS)
svi_sliders, svi_panel = build_weight_panel(SVI_DEFAULTS)

tab = widgets.Tab(children=[fhi_panel, svi_panel])
tab.set_title(0, '  FHI Weights (7 factors)  ')
tab.set_title(1, '  SVI Weights (4 factors)  ')
tab.layout = widgets.Layout(width='620px', margin='4px 0')

# ── Outputs and buttons ───────────────────────────────────────
static_out = widgets.Output()
si_out     = widgets.Output()
status_lbl = widgets.HTML(
    '<span style="color:#888;font-style:italic">Adjust weights then click Update Map.</span>'
)

update_btn = widgets.Button(
    description=' Update Map', button_style='success', icon='refresh',
    layout=widgets.Layout(width='145px', height='34px'),
)
save_btn = widgets.Button(
    description=' Save Map HTML', button_style='info', icon='download',
    tooltip='Export standalone HTML — embed in any website with <iframe>',
    layout=widgets.Layout(width='155px', height='34px'),
)

# Store the last computed arrays for the save function
_last = {'si': None, 'cats': None}


def compute(fhi_w, svi_w):
    fhi = sum(np.float32(fhi_w[n] / 100.0) * arr for n, arr in fhi_layers.items())
    svi = sum(np.float32(svi_w[n] / 100.0) * arr for n, arr in svi_layers.items())
    si  = np.float32(0.6) * (np.float32(6) - fhi) + \
          np.float32(0.4) * (np.float32(6) - svi)
    return fhi, svi, si


def on_update(b):
    update_btn.disabled = True
    save_btn.disabled   = True
    status_lbl.value = '<span style="color:darkorange;font-style:italic">Computing...</span>'

    fhi_w = {n: fhi_sliders[n].value for n in fhi_sliders}
    svi_w = {n: svi_sliders[n].value for n in svi_sliders}
    fhi, svi, si = compute(fhi_w, svi_w)
    cats = categorize_si(si)
    _last['si']   = si
    _last['cats'] = cats

    fig      = build_si_figure(si, cats)
    html_str = fig.to_html(full_html=False, include_plotlyjs='cdn')

    with si_out:
        si_out.clear_output(wait=True)
        display(IPHTML(html_str))

    with static_out:
        static_out.clear_output(wait=True)
        # FHI and SVI use auto-scaled colours (vmin/vmax=None) to match FHI_SVI.ipynb
        display(widgets.HBox([
            widgets.VBox([
                widgets.HTML('<h4 style="margin:4px 0">Flood Hazard Index</h4>'),
                mpl_png(fhi, 'Flood Hazard Index', 'YlOrRd',
                        'FHI Score (1=low, 5=high hazard)'),
            ]),
            widgets.VBox([
                widgets.HTML('<h4 style="margin:4px 0">Social Vulnerability Index</h4>'),
                mpl_png(svi, 'Social Vulnerability Index', 'YlOrRd',
                        'SVI Score (1=low, 5=high vulnerability)'),
            ]),
        ], layout=widgets.Layout(gap='24px')))

    del fhi, svi, fig, html_str
    gc.collect()
    status_lbl.value = '<span style="color:green;font-weight:600">Maps updated.</span>'
    update_btn.disabled = False
    save_btn.disabled   = False


def on_save(b):
    if _last['si'] is None:
        status_lbl.value = (
            '<span style="color:red">Run Update Map first.</span>'
        )
        return
    save_btn.disabled = True
    status_lbl.value  = '<span style="color:darkorange;font-style:italic">Saving...</span>'

    fig      = build_si_figure(_last['si'], _last['cats'])
    out_path = pathlib.Path(PROJECT_DIR) / 'suitability_map.html'
    # full_html=True + include_plotlyjs='cdn' → single self-contained file,
    # no internet needed for plotly.js after first load, easy <iframe> embed
    fig.write_html(
        str(out_path),
        full_html=True,
        include_plotlyjs='cdn',
        config={'responsive': True},
    )
    del fig
    gc.collect()
    status_lbl.value = (
        f'<span style="color:green;font-weight:600">Saved: {out_path}<br>'
        f'<span style="font-weight:normal;font-size:12px">'
        f'Embed in your site: '
        f'&lt;iframe src="suitability_map.html" width="100%" height="650px"&gt;'
        f'&lt;/iframe&gt;</span></span>'
    )
    save_btn.disabled = False


update_btn.on_click(on_update)
save_btn.on_click(on_save)

# ── Layout ────────────────────────────────────────────────────
display(widgets.VBox([
    tab,
    widgets.HBox(
        [update_btn, save_btn, status_lbl],
        layout=widgets.Layout(align_items='center', gap='10px', margin='10px 0 4px'),
    ),
    static_out,
    widgets.HTML('<h4 style="margin:12px 0 4px">Suitability Index — hover for cell score</h4>'),
    si_out,
]))